# LangGraph: Pecca as the agent's first tool

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pecca-core/pecca/blob/main/examples/notebooks/agent_langgraph.ipynb)

A minimal support agent has two tools: `classify_email` (an LLM-backed classifier) and `lookup_policy`. We run 20 emails,
then add Pecca's `route_rfi` as the **first** tool and run them again. The agent model is a scripted stand-in (no API keys, runs offline);
it calls `route_rfi` first and only uses the LLM classifier when Pecca says `fallback=True`.

In [1]:
%pip install -q pecca langgraph langchain-core

## Setup: stand-in LLM, tools, scripted model, and a trained Pecca model
*(boilerplate shared by all five agent notebooks)*

In [2]:
import ast, hashlib, json, os, re, tempfile

import pandas as pd
import pecca
from pecca.data.dataset import Dataset, canonicalize
from pecca.data.synthetic import CATALOG, RFIS, generate

work = tempfile.mkdtemp()
os.chdir(work)
os.environ["PECCA_HOME"] = os.path.join(work, ".pecca")

# --- A deterministic stand-in "LLM" for classification: keyword match, ~15% of answers wrong ------
_STOP = {"i", "my", "the", "a", "to", "me", "is", "and", "of", "on", "for", "in", "it", "do", "how", "can",
         "you", "please", "this", "that", "amt", "date", "last", "prod", "need", "want"}
_tok = lambda t: [w for w in re.findall(r"[a-z]+", t.lower()) if w not in _STOP]
_grams = lambda t: set(t) | {f"{a} {b}" for a, b in zip(t, t[1:])}
_TOPICS = {rfi: [_grams(_tok(p.replace("{", " ").replace("}", " "))) for p in ps]
           for rs in CATALOG.values() for rfi, ps in rs.items()}
LLM = {"calls": 0}                      # every real classification LLM call is counted here

def llm_classify(text):
    LLM["calls"] += 1
    g = _grams(_tok(text))
    best = max(_TOPICS, key=lambda r: (max(len(g & x) / len(x) for x in _TOPICS[r]), r))
    h = int(hashlib.sha256(text.encode()).hexdigest(), 16)
    return RFIS[h % len(RFIS)][1] if h % 100 < 15 else best

# --- The support agent's two ordinary tools -----------------------------------------------------
def classify_email(text: str) -> str:
    """Classify a customer email into one of 60 RFI topics (LLM-backed)."""
    return llm_classify(text)

def lookup_policy(topic: str) -> str:
    """Return the reply template for an RFI topic."""
    return f"Reply template for {topic.replace('_', ' ')}"

# --- The scripted "agent model" (no API key, fully offline) -------------------------------------
# It plans like a sensible model would: use route_rfi first if it exists; fall back to classify_email
# when Pecca says fallback=True (or when route_rfi is not available); then look up the policy.
TURNS = {"n": 0}                         # how many times the agent model was asked for its next step

def _parse(r):
    if isinstance(r, (dict, list)):
        return r
    for f in (json.loads, ast.literal_eval):
        try:
            return f(r)
        except Exception:
            pass
    return r

def plan(email, history, tools):
    """history: [(tool_name, result), ...]. Returns ('tool', name, args) or ('final', text)."""
    TURNS["n"] += 1
    done = {n: _parse(r) for n, r in history}
    if "route_rfi" in tools and "route_rfi" not in done:
        return ("tool", "route_rfi", {"text": email})
    if "route_rfi" in done and not done["route_rfi"]["fallback"]:
        topic = done["route_rfi"]["label"]
    elif "classify_email" in done:
        topic = done["classify_email"]
    else:
        return ("tool", "classify_email", {"text": email})
    if "lookup_policy" not in done:
        return ("tool", "lookup_policy", {"topic": topic})
    return ("final", f"[{topic}] {done['lookup_policy']}")

# --- Train Pecca on the logged LLM answers (+ human corrections) --------------------------------
df = generate(3000)
df["text"] = df.email_subject + "\n\n" + df.email_body
cols = {"input": "text", "llm_output": "llm_rfi", "human_label": "human_rfi", "call_name": {"literal": "route_rfi"}}
result = pecca.train("route_rfi", Dataset(canonicalize(df, cols), cols), candidates=["tfidf_linear"])
print(f"trained {result.winner}: macro_f1 {result.metric:.2f}, threshold {result.threshold:.2f}, "
      f"expected fallback {result.expected_fallback_rate:.0%}")

# 20 new emails to run through the agent
emails = generate(20, seed=7)
texts = (emails.email_subject + "\n\n" + emails.email_body).tolist()

trained tfidf_linear: macro_f1 0.92, threshold 0.59, expected fallback 0%


In [3]:
def report(run, tools_with_pecca, tools_without, label):
    LLM["calls"] = TURNS["n"] = 0
    base_out = run(tools_without)
    before = (LLM["calls"], TURNS["n"])
    LLM["calls"] = TURNS["n"] = 0
    pecca_out = run(tools_with_pecca)
    after = (LLM["calls"], TURNS["n"])
    print(f"{label}: 20 emails")
    print(f"  baseline (classify_email + lookup_policy):  classification LLM calls = {before[0]:>2}, agent model turns = {before[1]}")
    print(f"  with route_rfi first:                        classification LLM calls = {after[0]:>2}, agent model turns = {after[1]}")
    assert base_out == pecca_out or sum(a != b for a, b in zip(base_out, pecca_out)) <= 4   # same answers, apart from LLM noise
    return pecca_out

## The framework-specific part: a scripted chat model for LangGraph

In [4]:
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from pecca.integrations import langgraph as pecca_lg


class ScriptedModel(BaseChatModel):
    tool_names: list = []

    @property
    def _llm_type(self):
        return "scripted"

    def bind_tools(self, tools, **kwargs):
        return self.model_copy(update={"tool_names": [getattr(t, "name", str(t)) for t in tools]})

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        email = next(m.content for m in messages if isinstance(m, HumanMessage))
        history, called = [], {}
        for m in messages:
            if isinstance(m, AIMessage):
                called.update({c["id"]: c["name"] for c in m.tool_calls})
            if isinstance(m, ToolMessage):
                history.append((called[m.tool_call_id], m.content))
        step = plan(email, history, set(self.tool_names))
        msg = (AIMessage(content=step[1]) if step[0] == "final"
               else AIMessage(content="", tool_calls=[{"name": step[1], "args": step[2], "id": f"c{len(history)}"}]))
        return ChatResult(generations=[ChatGeneration(message=msg)])


def run(tools):
    agent = create_react_agent(ScriptedModel(), tools)
    return [agent.invoke({"messages": [("user", t)]})["messages"][-1].content for t in texts]

base_tools = [tool(classify_email), tool(lookup_policy)]

## (a)+(b) Baseline, then (c)+(d) with Pecca as the first tool

In [5]:
outs = report(run, [pecca_lg.tool("route_rfi")] + base_tools, base_tools, "LangGraph")

LangGraph: 20 emails
  baseline (classify_email + lookup_policy):  classification LLM calls = 20, agent model turns = 60
  with route_rfi first:                        classification LLM calls =  0, agent model turns = 60


## Per-email view: what Pecca decided
High confidence → Pecca answers (no LLM call); `fallback=True` → the agent falls back to the LLM classifier.

In [6]:
preds = pecca.predict("route_rfi", texts)
pd.DataFrame({
    "email": [t.split("\n\n")[0][:46] for t in texts],
    "pecca label": [p.label for p in preds],
    "confidence": [round(p.confidence, 2) for p in preds],
    "fallback to LLM": [p.fallback for p in preds],
    "agent answer": [o[:48] for o in outs],
})

,email,pecca label,confidence,fallback to LLM,agent answer
0,Please unlock my inactive account,dormant_account_reactivation,0.90,False,[dormant_account_reactivation] Reply template fo
1,I need a payment holiday on my loan,loan_payment_holiday,0.96,False,[loan_payment_holiday] Reply template for loan p
2,Can i repay my personal loan in full before th,loan_early_settlement,0.99,False,[loan_early_settlement] Reply template for loan
3,Re: What are the fees for an international tra,international_transfer_fees,0.98,False,[international_transfer_fees] Reply template for
4,I changed my address and need to update my det,account_details_update,0.95,False,[account_details_update] Reply template for acco
5,Re: How do i activate my new card,card_activation,0.95,False,[card_activation] Reply template for card activa
6,I would like to borrow more on my current loan,loan_top_up,0.98,False,[loan_top_up] Reply template for loan top up
7,"Please create a monthly standing order of $3,5",standing_order_setup,0.99,False,[standing_order_setup] Reply template for standi
8,Can you explain the interest on my loan of php,loan_interest_rate_query,0.99,False,[loan_interest_rate_query] Reply template for lo
9,A direct debit of $480 was taken in error,direct_debit_dispute,0.98,False,[direct_debit_dispute] Reply template for direct


## Takeaway
Adding `pecca.integrations.langgraph.tool("route_rfi")` as the first tool moved the cheap, deterministic routing step off the LLM:
classification LLM calls dropped while the agent still answers every email. The agent's own model turns are unchanged: Pecca replaces the *LLM-backed tool*, not the agent loop.